# Phase 3 — LegalIR strong rerankers on RTX Pro 6000 (offline)
Attach a competition-data dataset containing `train.json`, `private-official.json`, and `selected-contexts/selected-contexts/`, plus the Phase 2 Harrier bundle and Phase 3 reranker delta bundle. Set Internet Off. By default this runs private inference only; private Recall cannot be computed without labels. Its test-specific working directory is preserved when rerunning cells in the same live session so cached pipeline work can resume.

In [1]:
import os
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
from pathlib import Path
EXPERIMENT_ID = 'phase3-rerankers-harrier-retrieval'
DATASET_DIR = Path('/kaggle/input/datasets/tonioz/uit-dsc-task1')
TEST_FILENAME = 'private-official.json'  # switch to public-official.json for public inference
if TEST_FILENAME not in {'private-official.json', 'public-official.json'}: raise ValueError(f'Unsupported test input: {TEST_FILENAME}')
TEST_LABEL = 'private' if TEST_FILENAME == 'private-official.json' else 'public'
CHECKPOINT_ARTIFACTS_DIR = Path('/kaggle/input/datasets/boinhbo/artifacts-phase2/legalir-phase2-private-harrier-run/artifacts_phase2_harrier')
PHASE2_BUNDLE = Path('/kaggle/input/datasets/boinhbo/legalir-phase2-harrier-bundle/legalir-phase2-harrier-bundle')
DELTA_BUNDLE = Path('/kaggle/input/datasets/boinhbo/legalir-phase3-reranker-delta/legalir-phase3-reranker-delta')
WORK_DIR = Path(f'/kaggle/working/legalir-phase3-{TEST_LABEL}-run')
RUNTIME_DIR = Path('/kaggle/working/legalir-phase3-runtime')

In [2]:
import json, shutil, subprocess, sys, time
def run(*command, cwd=None, env=None):
    print('+', ' '.join(map(str, command)))
    started = time.perf_counter()
    subprocess.run(list(map(str, command)), cwd=cwd, env=env, check=True)
    print(f'Completed in {(time.perf_counter() - started) / 60:.1f} minutes')
delta_manifest = json.loads((DELTA_BUNDLE / 'manifests' / 'bundle_manifest.json').read_text(encoding='utf-8'))
if delta_manifest.get('experiment_id') != EXPERIMENT_ID: raise RuntimeError(f"Wrong Phase 3 bundle: {delta_manifest.get('experiment_id')}")
phase2_manifest_path = PHASE2_BUNDLE / 'manifests' / 'bundle_manifest.json'
if not phase2_manifest_path.is_file(): raise FileNotFoundError(f'Missing Phase 2 manifest: {phase2_manifest_path}')
phase2_manifest = json.loads(phase2_manifest_path.read_text(encoding='utf-8'))
phase2_names = {row['name'] for row in phase2_manifest['models']}
if not {'vietlegal_harrier', 'vietnamese_embedding', 'nemotron'} <= phase2_names: raise RuntimeError(f'Phase 2 bundle lacks retrievers: {phase2_names}')
for record in delta_manifest['files']:
    path = DELTA_BUNDLE / record['path']
    if not path.is_file() or path.stat().st_size != record['bytes']: raise RuntimeError(f'Missing/truncated delta file: {path}')
WORK_DIR.mkdir(parents=True, exist_ok=True)
if RUNTIME_DIR.exists(): shutil.rmtree(RUNTIME_DIR)
RUNTIME_DIR.mkdir(parents=True)
run(sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--ignore-installed', '--target', RUNTIME_DIR, '--find-links', DELTA_BUNDLE / 'wheels', '-r', DELTA_BUNDLE / 'requirements-offline.txt')
project_wheels = sorted((DELTA_BUNDLE / 'wheels').glob('uit_legalir-*.whl'))
if len(project_wheels) != 1: raise RuntimeError(f'Expected one project wheel, found {project_wheels}')
run(sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--ignore-installed', '--target', RUNTIME_DIR, project_wheels[0])
runtime_env = os.environ.copy()
runtime_env['PYTHONPATH'] = str(RUNTIME_DIR)
runtime_env['PYTHONNOUSERSITE'] = '1'
print('Phase 3 delta commit:', delta_manifest['project_commit'])

+ /usr/bin/python3 -m pip install --no-index --no-deps --ignore-installed --target /kaggle/working/legalir-phase3-runtime --find-links /kaggle/input/datasets/boinhbo/legalir-phase3-reranker-delta/legalir-phase3-reranker-delta/wheels -r /kaggle/input/datasets/boinhbo/legalir-phase3-reranker-delta/legalir-phase3-reranker-delta/requirements-offline.txt
Looking in links: /kaggle/input/datasets/boinhbo/legalir-phase3-reranker-delta/legalir-phase3-reranker-delta/wheels
Processing /kaggle/input/datasets/boinhbo/legalir-phase3-reranker-delta/legalir-phase3-reranker-delta/wheels/faiss_cpu-1.15.1-cp310-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (from -r /kaggle/input/datasets/boinhbo/legalir-phase3-reranker-delta/legalir-phase3-reranker-delta/requirements-offline.txt (line 4))
Processing /kaggle/input/datasets/boinhbo/legalir-phase3-reranker-delta/legalir-phase3-reranker-delta/wheels/sentence_transformers-5.7.0-py3-none-any.whl (from -r /kaggle/input/datasets/boinhbo/legalir-phase3-rer

In [3]:
gpu_probe = "import torch; assert torch.cuda.is_available(), 'No CUDA GPU available'; print('GPU:', torch.cuda.get_device_name(0)); print('CUDA:', torch.version.cuda)"
run(sys.executable, '-c', gpu_probe, env=runtime_env)
contexts_source = DATASET_DIR / 'selected-contexts' / 'selected-contexts'
if not contexts_source.is_dir(): raise FileNotFoundError(f'Missing corpus: {contexts_source}')
for filename in ('train.json', TEST_FILENAME):
    if not (DATASET_DIR / filename).is_file(): raise FileNotFoundError(f'Missing input: {filename}')
def ensure_input_link(destination, source, is_directory=False):
    if destination.is_symlink():
        if destination.resolve() == source.resolve(): return
        destination.unlink()
    elif destination.exists():
        raise RuntimeError(f'Refusing to overwrite existing work file: {destination}')
    destination.symlink_to(source, target_is_directory=is_directory)
ensure_input_link(WORK_DIR / 'selected-contexts', contexts_source, is_directory=True)
for filename in ('train.json', TEST_FILENAME): ensure_input_link(WORK_DIR / filename, DATASET_DIR / filename)
test_bytes = (WORK_DIR / TEST_FILENAME).read_bytes()
test_question_count = len(json.loads(test_bytes))
test_fingerprint = __import__('hashlib').sha256(test_bytes).hexdigest()
print(f"Contexts: {sum(1 for _ in contexts_source.glob('context_*.json'))}; {TEST_LABEL} questions from {TEST_FILENAME}: {test_question_count}")

+ /usr/bin/python3 -c import torch; assert torch.cuda.is_available(), 'No CUDA GPU available'; print('GPU:', torch.cuda.get_device_name(0)); print('CUDA:', torch.version.cuda)
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
CUDA: 12.8
Completed in 0.1 minutes
Contexts: 8532; private questions from private-official.json: 2080


In [4]:
import yaml
config = yaml.safe_load((DELTA_BUNDLE / 'configs' / 'kaggle_rtx_pro_6000.yaml').read_text(encoding='utf-8'))
config['paths']['public_file'] = TEST_FILENAME  # the pipeline calls this split 'public' internally
for name in ('vietlegal_harrier', 'vietnamese_embedding', 'nemotron'): config['models'][name]['local_path'] = str(PHASE2_BUNDLE / 'models' / name)
for name in ('legal_reranker', 'qwen3_reranker', 'prism_reranker'): config['models'][name]['local_path'] = str(DELTA_BUNDLE / 'models' / name)
for spec in config['models'].values(): spec['local_files_only'] = True
artifacts = WORK_DIR / config['paths']['artifacts_dir']
artifacts.mkdir(parents=True, exist_ok=True)
def seed_checkpoint(source_dir):
    if source_dir is None: return
    source_dir = Path(source_dir).resolve()
    if not source_dir.is_dir(): raise FileNotFoundError(f'Missing checkpoint artifacts: {source_dir}')
    prepare_checkpoint = source_dir / 'prepare_manifest.json'
    if not prepare_checkpoint.is_file(): raise RuntimeError(f'Checkpoint has no prepare manifest: {prepare_checkpoint}')
    prepare_metadata = json.loads(prepare_checkpoint.read_text(encoding='utf-8'))
    expected_chunking = __import__('hashlib').sha256(json.dumps(config['chunking'], ensure_ascii=False, sort_keys=True).encode('utf-8')).hexdigest()
    if prepare_metadata.get('chunking_fingerprint') != expected_chunking: raise RuntimeError('Checkpoint chunking config does not match this run')
    checkpoint_models = {}
    checkpoint_manifest = source_dir / 'model_manifest.json'
    if checkpoint_manifest.is_file(): checkpoint_models = {row['name']: row for row in json.loads(checkpoint_manifest.read_text(encoding='utf-8'))['models']}
    checkpoint_names = set(checkpoint_models)
    dense_names = {'vietlegal_harrier', 'vietnamese_embedding', 'nemotron'}
    for relative in ('corpus.jsonl', 'chunks_short.jsonl', 'chunks_long.jsonl', 'lexical_short.pkl'):
        source = source_dir / relative
        destination = artifacts / relative
        if source.exists() and not destination.exists(): destination.symlink_to(source, target_is_directory=source.is_dir())
    reused_dense = []
    rebuilt_dense = []
    for name in sorted(dense_names):
        spec = config['models'][name]
        record = checkpoint_models.get(name)
        model_matches = (not checkpoint_manifest.is_file()) if record is None else (record.get('id') == spec['id'] and record.get('revision') == spec.get('revision'))
        dense_source = source_dir / 'dense' / name
        dense_complete = model_matches and all((dense_source / required).is_file() for required in ('vectors.npy', 'index.faiss', 'chunks.json'))
        if dense_complete:
            destination = artifacts / 'dense' / name
            destination.parent.mkdir(parents=True, exist_ok=True)
            if not destination.exists(): destination.symlink_to(dense_source, target_is_directory=True)
            reused_dense.append(name)
        else:
            rebuilt_dense.append(name)
        memory_source = source_dir / 'question_memory' / name
        memory_complete = model_matches and all((memory_source / required).is_file() for required in ('vectors.npy', 'questions.json'))
        if memory_complete:
            destination = artifacts / 'question_memory' / name
            destination.parent.mkdir(parents=True, exist_ok=True)
            if not destination.exists(): destination.symlink_to(memory_source, target_is_directory=True)
    print('Reusable dense indexes:', reused_dense)
    print('Dense indexes to rebuild:', rebuilt_dense)
    phase3_names = set(config['models'])
    same_phase = checkpoint_names == phase3_names and not rebuilt_dense
    patterns = ['prepare_manifest.json', 'train_questions.jsonl']
    if same_phase: patterns += ['first_stage_weights.json', 'final_weights*.json', 'retrieval_train.json', 'fused_train.json', 'rerank_train*.json']
    for pattern in patterns:
        for source in source_dir.glob(pattern):
            destination = artifacts / source.name
            if not destination.exists(): shutil.copy2(source, destination)
    print('Seeded', 'Phase 3' if same_phase else 'compatible completed retrieval artifacts', 'checkpoint from:', source_dir)
seed_checkpoint(CHECKPOINT_ARTIFACTS_DIR)
input_state_path = WORK_DIR / 'inference_input_state.json'
config_fingerprint = __import__('hashlib').sha256(yaml.safe_dump(config, allow_unicode=True, sort_keys=True).encode('utf-8')).hexdigest()
input_state = {'filename': TEST_FILENAME, 'sha256': test_fingerprint, 'questions': test_question_count, 'config_sha256': config_fingerprint, 'project_commit': delta_manifest['project_commit']}
previous_input_state = json.loads(input_state_path.read_text(encoding='utf-8')) if input_state_path.is_file() else None
if previous_input_state != input_state:
    for pattern in ('public_questions.jsonl', 'retrieval_public.json', 'fused_public.json', 'rerank_public*.json'):
        for stale in artifacts.glob(pattern):
            if stale.is_file() or stale.is_symlink(): stale.unlink()
    print('Invalidated test-dependent caches:', previous_input_state, '->', input_state)
input_state_path.write_text(json.dumps(input_state, ensure_ascii=False, indent=2), encoding='utf-8')
config_path = WORK_DIR / 'kaggle_rtx_pro_6000_phase3.yaml'
config_path.write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False), encoding='utf-8')
print(config_path.read_text(encoding='utf-8'))

Reusable dense indexes: []
Dense indexes to rebuild: ['nemotron', 'vietlegal_harrier', 'vietnamese_embedding']
Seeded compatible completed retrieval artifacts checkpoint from: /kaggle/input/datasets/boinhbo/artifacts-phase2/legalir-phase2-private-harrier-run/artifacts_phase2_harrier
Invalidated test-dependent caches: None -> {'filename': 'private-official.json', 'sha256': '9da4e0cb84204fed924251c35744c93879556e67a440332015ea3b62f3c355bc', 'questions': 2080, 'config_sha256': '5de87bbe712fec84257bc98389c8f9657c2dffd9a0cb1f82661bbfa122bf75fb', 'project_commit': '944a51093314e299aa1f6855f0fdb8b9d2c293db'}
paths:
  contexts_dir: selected-contexts
  train_file: train.json
  public_file: private-official.json
  artifacts_dir: artifacts_phase3
  submission_file: submission_phase3_tuned.json
runtime:
  seed: 2026
  device: cuda
  dtype: bfloat16
  batch_size: 64
  num_workers: 4
models:
  vietlegal_harrier:
    id: mainguyen9/vietlegal-harrier-0.6b
    revision: 91a0e1ebe4b63b4475bbae40658b8ca9

In [5]:
preflight = WORK_DIR / 'phase3_preflight.py'
preflight.write_text('''
import sys
from pathlib import Path
import torch
import yaml
from legalir.embeddings import load_encoder
from legalir.rerank import PairwiseReranker, CausalYesNoReranker
config = yaml.safe_load(Path(sys.argv[1]).read_text(encoding='utf-8'))
for name, spec in config['models'].items():
    if spec['role'] == 'dense':
        model = load_encoder(spec, config['runtime'])
        assert len(model.encode([spec['prompt_query'] + 'điều kiện cấp giấy phép'], convert_to_numpy=True)) == 1
        del model
        torch.cuda.empty_cache()
for name, spec in config['models'].items():
    if spec['role'] == 'pairwise_reranker': engine = PairwiseReranker(config, name)
    elif spec['role'] == 'causal_reranker': engine = CausalYesNoReranker(config, name)
    else: continue
    assert len(engine.rank('câu hỏi pháp luật', ['văn bản pháp luật liên quan', 'văn bản không liên quan'])) == 2
    engine.close()
print('Phase 3 all local model preflight tests passed.')
'''.lstrip(), encoding='utf-8')
run(sys.executable, preflight, config_path, cwd=WORK_DIR, env=runtime_env)

+ /usr/bin/python3 /kaggle/working/legalir-phase3-private-run/phase3_preflight.py /kaggle/working/legalir-phase3-private-run/kaggle_rtx_pro_6000_phase3.yaml


Loading weights: 100%|██████████| 391/391 [00:03<00:00, 124.63it/s]
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'apply_yarn_scaling'}
Loading weights: 100%|██████████| 146/146 [00:00<00:00, 332.86it/s]
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'apply_yarn_scaling'}
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'apply_yarn_scaling'}
Loading weights: 100%|██████████| 320/320 [00:00<00:00, 649.16it/s]
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


Phase 3 all local model preflight tests passed.
Completed in 1.3 minutes


In [6]:
artifacts.mkdir(parents=True, exist_ok=True)
first_stage = {'weights': {'bm25': 0.0, 'accent_char': 0.5, 'vietlegal_harrier': 2.0, 'vietnamese_embedding': 1.0, 'nemotron': 2.0, 'query_memory': 1.0, 'query_exact': 4.0}, 'rrf_k': 20}
(artifacts / 'first_stage_weights.json').write_text(json.dumps(first_stage, ensure_ascii=False, indent=2), encoding='utf-8')
base = [sys.executable, '-m', 'legalir']
def legalir(*args): run(*base, *args, cwd=WORK_DIR, env=runtime_env)
legalir('prepare', '--config', config_path, '--resume')
legalir('audit', '--config', config_path)
legalir('index', '--config', config_path, '--lexical-only', '--resume')
for model in ('vietlegal_harrier', 'vietnamese_embedding', 'nemotron'): legalir('index', '--config', config_path, '--model', model, '--resume')
legalir('tune', '--config', config_path, '--final', '--fold', '0', '--resume')
shutil.copy2(artifacts / 'final_weights_fold0.json', artifacts / 'final_weights.json')
legalir('retrieve', '--config', config_path, '--split', 'public', '--resume')
for engine in ('legal_reranker', 'qwen3_reranker', 'prism_reranker'): legalir('rerank', '--config', config_path, '--split', 'public', '--engine', engine, '--resume')
legalir('rerank', '--config', config_path, '--split', 'public', '--resume')
submission_json = WORK_DIR / f'submission_phase3_{TEST_LABEL}_tuned.json'
submission_zip = WORK_DIR / f'submission_phase3_{TEST_LABEL}_tuned.zip'
legalir('predict', '--config', config_path, '--output', submission_json, '--resume')
run('zip', '-j', submission_zip, submission_json)
print('Submission:', submission_zip)

+ /usr/bin/python3 -m legalir prepare --config /kaggle/working/legalir-phase3-private-run/kaggle_rtx_pro_6000_phase3.yaml --resume


Preparing legal corpus: 100%|██████████| 8532/8532 [02:55<00:00, 48.67it/s]


{
  "schema_version": 2,
  "chunking_fingerprint": "58304a511c48e2ea202420ea24760f3591d5221d3140ffe7c258f57709e71b6c",
  "documents": 8532,
  "short_chunks": 420311,
  "long_chunks": 255832,
  "train_questions": 7000,
  "public_questions": 2080,
  "train_questions_fingerprint": "4a2f2a17b853138438c2b61c853f96637781909ce306bcd326a0213d53b4352e",
  "public_questions_fingerprint": "915600d911c468197dbf376a346767bfc527c781f3f5f7f56525898beba171e1"
}
Completed in 3.2 minutes
+ /usr/bin/python3 -m legalir audit --config /kaggle/working/legalir-phase3-private-run/kaggle_rtx_pro_6000_phase3.yaml


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 7987.56it/s]
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'apply_yarn_scaling'}
Loading weights: 100%|██████████| 320/320 [00:00<00:00, 7857.81it/s]


{
  "models": [
    {
      "name": "vietlegal_harrier",
      "id": "mainguyen9/vietlegal-harrier-0.6b",
      "parameters": 596049920,
      "revision": "91a0e1ebe4b63b4475bbae40658b8ca9231bea74",
      "license": "Apache-2.0"
    },
    {
      "name": "vietnamese_embedding",
      "id": "AITeamVN/Vietnamese_Embedding_v2",
      "parameters": 567754752,
      "revision": "18b44161e041bf1d3a333ab5144b5b7b93f914d2",
      "license": "Apache-2.0"
    },
    {
      "name": "nemotron",
      "id": "nvidia/Nemotron-3-Embed-1B-BF16",
      "parameters": 1140918272,
      "revision": "c0c9fea93ea424587517f2c59e20db9f1d6bf615",
      "license": "OpenMDW-1.1"
    },
    {
      "name": "legal_reranker",
      "id": "kiencnt2205/vietnamese-legal-reranker-bge-base",
      "parameters": 278044417,
      "revision": "e6894c39b03a2576972045c4fd3b9a2358eadb72",
      "license": "not-recorded"
    },
    {
      "name": "qwen3_reranker",
      "id": "Qwen/Qwen3-Reranker-0.6B",
      "parameters": 5

...

{
  "legal_reranker": {
    "44474": [
      "26173",
      "115380",
      "55018",
      "292631",
      "222640",
      "155884",
      "253656",
      "200618",
      "76646",
      "259205",
      "211665",
      "203327",
      "228779",
      "247532",
      "214592",
      "34136",
      "177389",
      "51323",
      "89199",
      "48646"
    ],
    "77938": [
      "298075",
      "52552",
      "146911",
      "136664",
      "35448",
      "210883",
      "174161",
      "136975",
      "53367",
      "281957",
      "134366",
      "226911",
      "59479",
      "59980",
      "98021",
      "98892",
      "128794",
      "126826",
      "280907",
      "122987"
    ],
    "33616": [
      "147066",
      "24778",
      "245154",
      "54776",
      "110305",
      "48429",
      "122159",
      "200446",
      "43981",
      "68399",
      "55840",
      "255913",
      "98892",
      "199484",
      "294760",
      "217349",
      "299985",
      "44264",
      "173530

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 14983.80it/s]


{
  "qwen3_reranker": {
    "44474": [
      "222640",
      "200618",
      "155884",
      "292631",
      "177389",
      "34136",
      "115380",
      "48646",
      "253656",
      "26173",
      "55018",
      "211665",
      "247532",
      "51323",
      "228779",
      "214592",
      "259205",
      "89199",
      "76646",
      "203327"
    ],
    "77938": [
      "136664",
      "52552",
      "210883",
      "35448",
      "298075",
      "146911",
      "53367",
      "281957",
      "59479",
      "134366",
      "136975",
      "174161",
      "128794",
      "280907",
      "226911",
      "98892",
      "59980",
      "122987",
      "126826",
      "98021"
    ],
    "33616": [
      "24778",
      "147066",
      "48429",
      "110305",
      "122159",
      "245154",
      "205761",
      "54776",
      "199484",
      "98892",
      "299985",
      "68399",
      "55840",
      "255913",
      "294760",
      "43981",
      "200446",
      "173530",
      "44264

Loading weights: 100%|██████████| 320/320 [00:00<00:00, 14668.44it/s]
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


{
  "prism_reranker": {
    "44474": [
      "292631",
      "155884",
      "34136",
      "48646",
      "89199",
      "203327",
      "51323",
      "115380",
      "253656",
      "55018",
      "214592",
      "259205",
      "76646",
      "228779",
      "200618",
      "247532",
      "177389",
      "222640",
      "211665",
      "26173"
    ],
    "77938": [
      "281957",
      "35448",
      "146911",
      "136975",
      "52552",
      "210883",
      "53367",
      "134366",
      "59479",
      "174161",
      "128794",
      "98021",
      "59980",
      "280907",
      "226911",
      "126826",
      "98892",
      "298075",
      "122987",
      "136664"
    ],
    "33616": [
      "245154",
      "255913",
      "173530",
      "217349",
      "205761",
      "98892",
      "147066",
      "24778",
      "199484",
      "54776",
      "48429",
      "110305",
      "68399",
      "294760",
      "299985",
      "55840",
      "44264",
      "43981",
      "200446

In [7]:
manifest = json.loads((artifacts / 'model_manifest.json').read_text(encoding='utf-8'))
final_stage = json.loads((artifacts / 'final_weights.json').read_text(encoding='utf-8'))
report = {'experiment_id': EXPERIMENT_ID, 'test_file': TEST_FILENAME, 'test_questions': test_question_count, 'test_sha256': test_fingerprint, 'evaluation': 'inference_only; labels and Recall not evaluated', 'project_commit': delta_manifest['project_commit'], 'models': manifest['models'], 'total_parameters': manifest['total_parameters'], 'first_stage': first_stage, 'final_stage': final_stage, 'submission': str(submission_zip)}
(WORK_DIR / 'phase3_report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(report, ensure_ascii=False, indent=2))

{
  "experiment_id": "phase3-rerankers-harrier-retrieval",
  "test_file": "private-official.json",
  "test_questions": 2080,
  "test_sha256": "9da4e0cb84204fed924251c35744c93879556e67a440332015ea3b62f3c355bc",
  "evaluation": "inference_only; labels and Recall not evaluated",
  "project_commit": "944a51093314e299aa1f6855f0fdb8b9d2c293db",
  "models": [
    {
      "name": "vietlegal_harrier",
      "id": "mainguyen9/vietlegal-harrier-0.6b",
      "parameters": 596049920,
      "revision": "91a0e1ebe4b63b4475bbae40658b8ca9231bea74",
      "license": "Apache-2.0"
    },
    {
      "name": "vietnamese_embedding",
      "id": "AITeamVN/Vietnamese_Embedding_v2",
      "parameters": 567754752,
      "revision": "18b44161e041bf1d3a333ab5144b5b7b93f914d2",
      "license": "Apache-2.0"
    },
    {
      "name": "nemotron",
      "id": "nvidia/Nemotron-3-Embed-1B-BF16",
      "parameters": 1140918272,
      "revision": "c0c9fea93ea424587517f2c59e20db9f1d6bf615",
      "license": "OpenMDW-1.1"